In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://books.toscrape.com/"

response = requests.get(url)
response.raise_for_status()

response.encoding = "utf-8"

soup = BeautifulSoup(response.text, "html.parser")
books = soup.find_all("article", class_="product_pod")

len(books)

In [ ]:
book = books[0]
book.h3.a["title"], book.find("p", class_="price_color").text

In [ ]:
import pandas as pd

data = []

for book in books:
    title = book.h3.a["title"]
    price = book.find("p", class_="price_color").text

    data.append({
        "title": title,
        "price": price
    })

df = pd.DataFrame(data)

df.head()

In [ ]:
print(book.prettify())

In [ ]:
for tag in book.find_all():
    print(
        "TAG:", tag.name,
        "| ATTRS:", tag.attrs,
        "| TEXT:", tag.get_text(" ", strip=True)
    )

In [2]:
import requests
from bs4 import BeautifulSoup

url = "https://www.unr.edu/admissions/records/academic-calendar/finals-schedule"

response = requests.get(url)
response.raise_for_status()

print(response.status_code)
print(response.encoding)



200
utf-8


In [3]:
soup = BeautifulSoup(response.text, "html.parser")

In [4]:
soup.title

<title>Generic Finals Schedule | Scheduling Services | University of Nevada, Reno</title>

In [ ]:
soup.find_all("table")

In [6]:
tables = soup.find_all("table")
len(tables)

6

In [7]:
print(tables[0].prettify())

<table class="footable">
 <caption>
  Thursday, first day of finals
 </caption>
 <thead>
  <tr>
   <th scope="col">
    Class time
   </th>
   <th class="breakpoint-sm" scope="col">
    Class day(s)
   </th>
   <th class="breakpoint-sm" scope="col">
    Final meeting time
   </th>
  </tr>
 </thead>
 <tbody>
  <tr>
   <th scope="row">
    9 a.m.
   </th>
   <td>
    Tuesday/Thursday (TR) or Thursday (R)
   </td>
   <td>
    8-10 a.m.
   </td>
  </tr>
  <tr>
   <th scope="row">
    12 p.m.
   </th>
   <td>
    Tuesday/Thursday (TR)
   </td>
   <td>
    10:15 a.m.-12:15 p.m.
   </td>
  </tr>
  <tr>
   <th scope="row">
    3 p.m.
   </th>
   <td>
    Tuesday/Thursday (TR) or Thursday (R)
   </td>
   <td>
    3-5 p.m.
   </td>
  </tr>
  <tr>
   <th scope="row">
    4:30 p.m.
   </th>
   <td>
    Thursday (R)
   </td>
   <td>
    5:30-7:30 p.m.
   </td>
  </tr>
  <tr>
   <th scope="row">
    6 p.m.
   </th>
   <td>
    Tuesday/Thursday (TR) or Thursday (R)
   </td>
   <td>
    5:30-7:30 p.m.

In [8]:
for tag in soup.find_all(["h1", "h2", "h3"]):
    print(tag.name, ":", tag.get_text(" ", strip=True))

h2 : Quicklinks
h2 : Search
h1 : Finals schedule
h2 : Wednesday, prep day
h2 : Thursday
h2 : Friday
h2 : Saturday
h2 : Monday
h2 : Tuesday
h2 : Wednesday


In [9]:
table = tables[0]

rows = table.find_all("tr")

len(rows)

8

In [10]:
print(rows[0].prettify())

<tr>
 <th scope="col">
  Class time
 </th>
 <th class="breakpoint-sm" scope="col">
  Class day(s)
 </th>
 <th class="breakpoint-sm" scope="col">
  Final meeting time
 </th>
</tr>



In [11]:
tables[0].get("class")

['footable']

In [12]:
table = tables[0]

finals_day = table.caption.get_text(" ", strip=True)
finals_day

'Thursday, first day of finals'

In [13]:
row = table.tbody.find_all("tr")[0]

class_time = row.find("th", scope="row").get_text(" ", strip=True)
cells = row.find_all("td")

class_days = cells[0].get_text(" ", strip=True)
final_meeting = cells[1].get_text(" ", strip=True)

class_time, class_days, final_meeting

('9 a.m.', 'Tuesday/Thursday (TR) or Thursday (R)', '8-10 a.m.')

In [17]:
data = []

for table in tables:
    finals_day = table.caption.get_text(" ", strip=True)

    for row in table.tbody.find_all("tr"):
        class_time = row.find("th", scope="row").get_text(" ", strip=True)

        cells = row.find_all("td")

        class_days = cells[0].get_text(" ", strip=True)
        final_meeting = cells[1].get_text(" ", strip=True)

        data.append({
            "Class Day(s)": class_days,
            "Class time": class_time,
            "Finals Day": finals_day,
            "Final Meeting time": final_meeting
        })

In [20]:
import pandas as pd

df = pd.DataFrame(data)

df["Class Day(s)"] = df["Class Day(s)"].str.split(" or ")

df = df.explode("Class Day(s)", ignore_index=True)



In [32]:
df[df["Class Day(s)"] == "Monday/Wednesday (MW)"]

,Class Day(s),Class time,Finals Day,Final Meeting time
11,Monday/Wednesday (MW),8 a.m.,"Friday, second day of finals",8-10 a.m.
23,Monday/Wednesday (MW),10 a.m.,"Monday, third day of finals",10:15 a.m.-12:15 p.m.
24,Monday/Wednesday (MW),1 p.m.,"Monday, third day of finals",12:45-2:45 p.m.
25,Monday/Wednesday (MW),2:30 p.m.,"Monday, third day of finals",3-5 p.m.
27,Monday/Wednesday (MW),5:30 p.m.,"Monday, third day of finals",5:30-7:30 p.m.
29,Monday/Wednesday (MW),7 p.m.,"Monday, third day of finals",7:45-9:45 p.m.
42,Monday/Wednesday (MW),9 a.m.,"Wednesday, fifth day of finals",8-10 a.m.
44,Monday/Wednesday (MW),11 a.m.,"Wednesday, fifth day of finals",10:15 a.m.-12:15 p.m.
46,Monday/Wednesday (MW),12 p.m.,"Wednesday, fifth day of finals",12:45-2:45 p.m.
47,Monday/Wednesday (MW),4 p.m.,"Wednesday, fifth day of finals",3-5 p.m.


In [33]:
df[df["Class Day(s)"] == "Tuesday/Thursday (TR)"]

,Class Day(s),Class time,Finals Day,Final Meeting time
0,Tuesday/Thursday (TR),9 a.m.,"Thursday, first day of finals",8-10 a.m.
2,Tuesday/Thursday (TR),12 p.m.,"Thursday, first day of finals",10:15 a.m.-12:15 p.m.
3,Tuesday/Thursday (TR),3 p.m.,"Thursday, first day of finals",3-5 p.m.
6,Tuesday/Thursday (TR),6 p.m.,"Thursday, first day of finals",5:30-7:30 p.m.
9,Tuesday/Thursday (TR),9 p.m.,"Thursday, first day of finals",7:45-9:45 p.m.
31,Tuesday/Thursday (TR),7:30 a.m.,"Tuesday, fourth day of finals",8-10 a.m.
32,Tuesday/Thursday (TR),9:30 a.m.,"Tuesday, fourth day of finals",10:15 a.m.-12:15 p.m.
33,Tuesday/Thursday (TR),10:30 a.m.,"Tuesday, fourth day of finals",10:15 a.m.-12:15 p.m.
34,Tuesday/Thursday (TR),1:30 p.m.,"Tuesday, fourth day of finals",12:45-2:45 p.m.
36,Tuesday/Thursday (TR),4:30 p.m.,"Tuesday, fourth day of finals",3-5 p.m.


In [21]:
df.head(20)

,Class Day(s),Class time,Finals Day,Final Meeting time
0,Tuesday/Thursday (TR),9 a.m.,"Thursday, first day of finals",8-10 a.m.
1,Thursday (R),9 a.m.,"Thursday, first day of finals",8-10 a.m.
2,Tuesday/Thursday (TR),12 p.m.,"Thursday, first day of finals",10:15 a.m.-12:15 p.m.
3,Tuesday/Thursday (TR),3 p.m.,"Thursday, first day of finals",3-5 p.m.
4,Thursday (R),3 p.m.,"Thursday, first day of finals",3-5 p.m.
5,Thursday (R),4:30 p.m.,"Thursday, first day of finals",5:30-7:30 p.m.
6,Tuesday/Thursday (TR),6 p.m.,"Thursday, first day of finals",5:30-7:30 p.m.
7,Thursday (R),6 p.m.,"Thursday, first day of finals",5:30-7:30 p.m.
8,Thursday (R),7:30 p.m.,"Thursday, first day of finals",7:45-9:45 p.m.
9,Tuesday/Thursday (TR),9 p.m.,"Thursday, first day of finals",7:45-9:45 p.m.


In [22]:
df.tail(10)

,Class Day(s),Class time,Finals Day,Final Meeting time
41,Monday/Wednesday/Friday (MWF),9 a.m.,"Wednesday, fifth day of finals",8-10 a.m.
42,Monday/Wednesday (MW),9 a.m.,"Wednesday, fifth day of finals",8-10 a.m.
43,Monday/Wednesday/Friday (MWF),11 a.m.,"Wednesday, fifth day of finals",10:15 a.m.-12:15 p.m.
44,Monday/Wednesday (MW),11 a.m.,"Wednesday, fifth day of finals",10:15 a.m.-12:15 p.m.
45,Monday/Wednesday/Friday (MWF),12 p.m.,"Wednesday, fifth day of finals",12:45-2:45 p.m.
46,Monday/Wednesday (MW),12 p.m.,"Wednesday, fifth day of finals",12:45-2:45 p.m.
47,Monday/Wednesday (MW),4 p.m.,"Wednesday, fifth day of finals",3-5 p.m.
48,Wednesday (W),4 p.m.,"Wednesday, fifth day of finals",3-5 p.m.
49,Wednesday (W),5:30 p.m.,"Wednesday, fifth day of finals",5:30-7:30 p.m.
50,Wednesday (W),7 p.m.,"Wednesday, fifth day of finals",5:30-7:30 p.m.


In [26]:
data = {

    "Course" : ["CS457", "Stat654", "Math311", "Math466"],
    "Day"    : [
                "Monday/Wednesday (MW)", 
                "Tuesday/Thursday (TR)", 
                "Monday/Wednesday/Friday (MWF)", 
                "Monday/Wednesday/Friday (MWF)"],
    "Time" : ["7 p.m.", "10:30 a.m.", "9 a.m.", "12 p.m."]

}

courses = pd.DataFrame(data)

courses.head()

,Course,Day,Time
0,CS457,Monday/Wednesday (MW),7 p.m.
1,Stat654,Tuesday/Thursday (TR),10:30 a.m.
2,Math311,Monday/Wednesday/Friday (MWF),9 a.m.
3,Math466,Monday/Wednesday/Friday (MWF),12 p.m.


In [24]:
df["Class time"].unique()

<StringArray>
[    '9 a.m.',    '12 p.m.',     '3 p.m.',  '4:30 p.m.',     '6 p.m.',
  '7:30 p.m.',     '9 p.m.',     '8 a.m.',     '1 p.m.',  '2:30 p.m.',
     '4 p.m.',  '5:30 p.m.',     '7 p.m.', '*Conflicts',     '7 a.m.',
    '10 a.m.',  '7:30 a.m.',  '9:30 a.m.', '10:30 a.m.',  '1:30 p.m.',
    '11 a.m.']
Length: 21, dtype: str

In [27]:
courses["Time"].unique()

<StringArray>
['7 p.m.', '10:30 a.m.', '9 a.m.', '12 p.m.']
Length: 4, dtype: str

In [28]:
result = courses.merge(
    df,
    left_on=["Day", "Time"],
    right_on=["Class Day(s)", "Class time"],
    how="left"
)

In [29]:
result

,Course,Day,Time,Class Day(s),Class time,Finals Day,Final Meeting time
0,CS457,Monday/Wednesday (MW),7 p.m.,Monday/Wednesday (MW),7 p.m.,"Monday, third day of finals",7:45-9:45 p.m.
1,Stat654,Tuesday/Thursday (TR),10:30 a.m.,Tuesday/Thursday (TR),10:30 a.m.,"Tuesday, fourth day of finals",10:15 a.m.-12:15 p.m.
2,Math311,Monday/Wednesday/Friday (MWF),9 a.m.,Monday/Wednesday/Friday (MWF),9 a.m.,"Wednesday, fifth day of finals",8-10 a.m.
3,Math466,Monday/Wednesday/Friday (MWF),12 p.m.,Monday/Wednesday/Friday (MWF),12 p.m.,"Wednesday, fifth day of finals",12:45-2:45 p.m.


In [30]:
result = result.drop(
    columns=["Class Day(s)", "Class time"]
)

In [31]:
result

,Course,Day,Time,Finals Day,Final Meeting time
0,CS457,Monday/Wednesday (MW),7 p.m.,"Monday, third day of finals",7:45-9:45 p.m.
1,Stat654,Tuesday/Thursday (TR),10:30 a.m.,"Tuesday, fourth day of finals",10:15 a.m.-12:15 p.m.
2,Math311,Monday/Wednesday/Friday (MWF),9 a.m.,"Wednesday, fifth day of finals",8-10 a.m.
3,Math466,Monday/Wednesday/Friday (MWF),12 p.m.,"Wednesday, fifth day of finals",12:45-2:45 p.m.
